In [2]:
import torch
from torch import nn
import pandas as pd
import numpy as np

In [6]:
# dataset: https://www.kaggle.com/datasets/mragpavank/insurance1

df = pd.get_dummies(pd.read_csv('insurance.csv'))
df.head()

,age,bmi,children,charges,sex_female,sex_male,smoker_no,smoker_yes,region_northeast,region_northwest,region_southeast,region_southwest
0,19,27.900,0,16884.92400,True,False,False,True,False,False,False,True
1,18,33.770,1,1725.55230,False,True,True,False,False,False,True,False
2,28,33.000,3,4449.46200,False,True,True,False,False,False,True,False
3,33,22.705,0,21984.47061,False,True,True,False,False,True,False,False
4,32,28.880,0,3866.85520,False,True,True,False,False,True,False,False


In [7]:
x, y = df.drop('charges', axis=1), df['charges']

In [8]:
train_split = int(len(df) * 0.8)

x_train = x[:train_split].to_numpy(dtype=np.float32)
y_train = y[:train_split].to_numpy(dtype=np.float32) / y.max()

x_test = x[train_split:].to_numpy(dtype=np.float32)
y_test = y[train_split:].to_numpy(dtype=np.float32) / y.max()

In [9]:
x_train = torch.from_numpy(x_train)
y_train = torch.from_numpy(y_train).unsqueeze(1)

x_test = torch.from_numpy(x_test)
y_test = torch.from_numpy(y_test).unsqueeze(1)

In [10]:
model = nn.Linear(in_features=x_train.shape[1], out_features=1)

In [11]:
loss_fn = nn.L1Loss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.01)

In [13]:
for epoch in range(101):

    model.train()
    optimizer.zero_grad()

    y_pred = model(x_train)
    loss = loss_fn(y_pred, y_train)

    loss.backward()
    optimizer.step()

    if (epoch % 10 == 0):
        model.eval()

        with torch.no_grad():
            y_pred_test = model(x_test)
            test_loss = loss_fn(y_pred_test, y_test)

            print(f'{epoch}: train_loss = {loss.item()}, test_loss = {test_loss}')

        model.train()


0: train_loss = 0.0750468447804451, test_loss = 0.09365168958902359
10: train_loss = 0.09176554530858994, test_loss = 0.1348373144865036
20: train_loss = 0.08973599970340729, test_loss = 0.0849883109331131
30: train_loss = 0.12374291568994522, test_loss = 0.08170633018016815
40: train_loss = 0.07588398456573486, test_loss = 0.12017961591482162
50: train_loss = 0.10771552473306656, test_loss = 0.11270837485790253
60: train_loss = 0.08937394618988037, test_loss = 0.09856247156858444
70: train_loss = 0.0973920002579689, test_loss = 0.08292131870985031
80: train_loss = 0.07646684348583221, test_loss = 0.07695789635181427
90: train_loss = 0.09551084786653519, test_loss = 0.06310303509235382
100: train_loss = 0.07653291523456573, test_loss = 0.07669323682785034


In [18]:
torch.save(model.state_dict(), 'Medical_Cost.pth')